### Lab5
### Ньяти Каелиле

In [22]:
# Cell 1: Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


### Exercise 1

In [23]:
# Cell 2: Define the two-channel queuing system simulation function
def simulate_two_channel_system(lambda_rate, mu_rate, T, verbose=False):
    """
    Simulate a two-channel queuing system with failures
    
    Parameters:
    lambda_rate: arrival rate (λ) [1/sec]
    mu_rate: service rate (μ) [1/sec]
    T: total simulation time [sec]
    verbose: if True, print detailed simulation information
    
    Returns:
    Dictionary with simulation results
    """
    
    # Initialize variables
    t = 0  # current time
    m = 0  # number of processed requests
    n_rejections = 0  # number of rejected requests
    
    # Channel state variables
    # First channel
    Ts1_b = 0  # start time of current service
    Ts1_e = 0  # end time of current service
    idle_time1 = 0  # total idle time for channel 1
    
    # Second channel
    Ts2_b = 0  # start time of current service
    Ts2_e = 0  # end time of current service
    idle_time2 = 0  # total idle time for channel 2
    
    # Generate first arrival
    p = np.random.random()
    t_arrival = (-1/lambda_rate) * np.log(p)
    
    # Simulation loop
    while t_arrival < T:
        if verbose:
            print(f"\nTime: {t_arrival:.3f}")
        
        # Check which channels are free at arrival time
        channel1_free = (t_arrival >= Ts1_e)
        channel2_free = (t_arrival >= Ts2_e)
        
        if channel1_free:
            # Channel 1 is free - assign request to channel 1
            if Ts1_e > 0:  # Accumulate idle time if this is not the first request
                idle_time1 += t_arrival - Ts1_e
            
            Ts1_b = t_arrival
            p = np.random.random()
            service_time = (-1/mu_rate) * np.log(p)
            Ts1_e = Ts1_b + service_time
            m += 1
            
            if verbose:
                print(f"Request assigned to Channel 1, service time: {service_time:.3f}")
        
        elif channel2_free:
            # Channel 2 is free - assign request to channel 2
            if Ts2_e > 0:  # Accumulate idle time if this is not the first request
                idle_time2 += t_arrival - Ts2_e
            
            Ts2_b = t_arrival
            p = np.random.random()
            service_time = (-1/mu_rate) * np.log(p)
            Ts2_e = Ts2_b + service_time
            m += 1
            
            if verbose:
                print(f"Request assigned to Channel 2, service time: {service_time:.3f}")
        
        else:
            # Both channels busy - request rejected
            n_rejections += 1
            if verbose:
                print("Request rejected - both channels busy")
        
        # Generate next arrival
        p = np.random.random()
        interarrival_time = (-1/lambda_rate) * np.log(p)
        t_arrival += interarrival_time
    
    # Calculate final idle times
    if Ts1_e < T:
        idle_time1 += T - Ts1_e
    if Ts2_e < T:
        idle_time2 += T - Ts2_e
    
    # Calculate probabilities and statistics
    total_requests = m + n_rejections
    
    # Q - Relative throughput (probability of service)
    Q = m / total_requests if total_requests > 0 else 0
    
    # A - Absolute throughput (average number of processed requests per second)
    A = m / T
    
    # Probabilities of idle time for each channel
    p_idle1 = idle_time1 / T
    p_idle2 = idle_time2 / T
    
    # k - Load coefficient (total utilization of both channels)
    k = (T - idle_time1 + T - idle_time2) / T  # Total busy time / T
    
    results = {
        'processed': m,
        'rejected': n_rejections,
        'total_requests': total_requests,
        'p_success': Q,  # This is Q - relative throughput
        'p_reject': n_rejections / total_requests if total_requests > 0 else 0,
        'Q': Q,  # Relative throughput'A': A,  # Absolute throughput
        'A': A,
        'k': k,  # Load coefficient
        'idle_time1': idle_time1,
        'idle_time2': idle_time2,
        'p_idle1': p_idle1,
        'p_idle2': p_idle2,
        'p_busy1': 1 - p_idle1,
        'p_busy2': 1 - p_idle2,
        'p_busy_any': 1 - (p_idle1 * p_idle2)
    }
    
    return results

In [24]:
# Cell 3: Define analytical calculations for two-channel M/M/2 system
def analytical_results(lambda_rate, mu_rate):
    """
    Calculate analytical results for two-channel queuing system
    
    Parameters:
    lambda_rate: arrival rate (λ) [1/sec]
    mu_rate: service rate (μ) [1/sec]
    
    Returns:
    Dictionary with analytical results
    """
    
    rho = lambda_rate / mu_rate  # Load factor
    
    # Probability system is idle
    P0 = 1 / (1 + rho + (rho**2)/2)
    
    # Probability of rejection (both channels busy)
    Pf = (rho**2 / 2) * P0
    
    # Relative throughput (probability of service)
    Q = 1 - Pf
    
    # Absolute throughput
    A = Q * lambda_rate
    
    # Channel load factor
    k = A / mu_rate
    
    results = {
        'rho': rho,
        'P0': P0,
        'Pf': Pf,
        'Q': Q,
        'A': A,
        'k': k
    }
    
    return results

### Exercise 2

In [25]:
# Cell 4: ЗАДАНИЕ №2 - Расчет для заданных параметров
print("=" * 70)
print("ЗАДАНИЕ №2: Моделирование двухканальной системы с отказами")
print("=" * 70)

# Input parameters
mu = 0.1  # service rate [1/sec]
lambda_rate = 0.2  # arrival rate [1/sec]
T = 1000  # simulation time [sec]

print(f"Исходные данные:")
print(f"Интенсивность обработки заявок (μ) = {mu} [1/сек]")
print(f"Интенсивность входного потока (λ) = {lambda_rate} [1/сек]")
print(f"Время моделирования (T) = {T} [сек]")
print()

# Run simulation
results = simulate_two_channel_system(lambda_rate, mu, T, verbose=False)

print("=" * 70)
print("РЕЗУЛЬТАТЫ МОДЕЛИРОВАНИЯ:")
print("=" * 70)
print(f"\n1. Основные показатели:")
print(f"   - Число обработанных заявок (m): {results['processed']}")
print(f"   - Число отказов: {results['rejected']}")
print(f"   - Всего заявок: {results['total_requests']}")

print(f"\n2. Пропускная способность:")
print(f"   - Q (Относительная пропускная способность): {results['Q']:.4f}")
print(f"   - A (Абсолютная пропускная способность): {results['A']:.4f} [1/сек]")
print(f"   - k (Коэффициент загрузки каналов): {results['k']:.4f}")

print(f"\n3. Вероятности:")
print(f"   - Вероятность обработки заявки (Q): {results['p_success']:.4f}")
print(f"   - Вероятность отказа в обслуживании: {results['p_reject']:.4f}")

print(f"\n4. Временные характеристики:")
print(f"   - Время простоя первого канала: {results['idle_time1']:.2f} сек")
print(f"   - Время простоя второго канала: {results['idle_time2']:.2f} сек")
print(f"   - Вероятность простоя первого канала: {results['p_idle1']:.4f}")
print(f"   - Вероятность простоя второго канала: {results['p_idle2']:.4f}")

print(f"\n5. Загрузка каналов:")
print(f"   - Вероятность загрузки первого канала: {results['p_busy1']:.4f}")
print(f"   - Вероятность загрузки второго канала: {results['p_busy2']:.4f}")
print(f"   - Вероятность загрузки первого или второго каналов: {results['p_busy_any']:.4f}")

ЗАДАНИЕ №2: Моделирование двухканальной системы с отказами
Исходные данные:
Интенсивность обработки заявок (μ) = 0.1 [1/сек]
Интенсивность входного потока (λ) = 0.2 [1/сек]
Время моделирования (T) = 1000 [сек]

РЕЗУЛЬТАТЫ МОДЕЛИРОВАНИЯ:

1. Основные показатели:
   - Число обработанных заявок (m): 118
   - Число отказов: 82
   - Всего заявок: 200

2. Пропускная способность:
   - Q (Относительная пропускная способность): 0.5900
   - A (Абсолютная пропускная способность): 0.1180 [1/сек]
   - k (Коэффициент загрузки каналов): 1.3014

3. Вероятности:
   - Вероятность обработки заявки (Q): 0.5900
   - Вероятность отказа в обслуживании: 0.4100

4. Временные характеристики:
   - Время простоя первого канала: 271.57 сек
   - Время простоя второго канала: 427.03 сек
   - Вероятность простоя первого канала: 0.2716
   - Вероятность простоя второго канала: 0.4270

5. Загрузка каналов:
   - Вероятность загрузки первого канала: 0.7284
   - Вероятность загрузки второго канала: 0.5730
   - Вероятность 

### Exercise 3.1

In [26]:
# Cell 5: ЗАДАНИЕ №3.1 - Сопоставление с аналитическими результатами
print("=" * 70)
print("ЗАДАНИЕ №3.1: Сопоставление с аналитическими результатами")
print("=" * 70)

# Calculate analytical results
analytical = analytical_results(lambda_rate, mu)

print("\nАНАЛИТИЧЕСКИЕ РЕЗУЛЬТАТЫ (на основе теории массового обслуживания):")
print("-" * 50)
print(f"Коэффициент загрузки системы (ρ = λ/μ) = {analytical['rho']:.4f}")
print(f"Вероятность простоя системы (P0) = {analytical['P0']:.4f}")
print(f"Вероятность отказа в обслуживании (Pf) = {analytical['Pf']:.4f}")
print(f"Q (Относительная пропускная способность) = {analytical['Q']:.4f}")
print(f"A (Абсолютная пропускная способность) = {analytical['A']:.4f} [1/сек]")
print(f"k (Коэффициент загрузки каналов) = {analytical['k']:.4f}")

print("\n" + "=" * 70)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ:")
print("=" * 70)
print(f"{'Показатель':<35} {'Моделирование':<15} {'Аналитика':<15} {'Отклонение':<15}")
print("-" * 80)

# Calculate relative errors
error_Q = abs(results['Q'] - analytical['Q']) / analytical['Q'] * 100
error_A = abs(results['A'] - analytical['A']) / analytical['A'] * 100
error_k = abs(results['k'] - analytical['k']) / analytical['k'] * 100
error_Pf = abs(results['p_reject'] - analytical['Pf']) / analytical['Pf'] * 100

print(f"{'Q (Относительная пропускная способность)':<35} {results['Q']:<15.4f} {analytical['Q']:<15.4f} {error_Q:<14.2f}%")
print(f"{'A (Абсолютная пропускная способность)':<35} {results['A']:<15.4f} {analytical['A']:<15.4f} {error_A:<14.2f}%")
print(f"{'k (Коэффициент загрузки каналов)':<35} {results['k']:<15.4f} {analytical['k']:<15.4f} {error_k:<14.2f}%")
print(f"{'Вероятность отказа (Pf)':<35} {results['p_reject']:<15.4f} {analytical['Pf']:<15.4f} {error_Pf:<14.2f}%")
print(f"{'Вероятность обработки':<35} {results['p_success']:<15.4f} {analytical['Q']:<15.4f} {error_Q:<14.2f}%")

print("\n" + "=" * 70)
print("ВЫВОД:")
print("=" * 70)
print(f"Относительная погрешность моделирования составляет:")
print(f"  - Для Q: {error_Q:.2f}%")
print(f"  - Для A: {error_A:.2f}%")
print(f"  - Для k: {error_k:.2f}%")
print(f"  - Для Pf: {error_Pf:.2f}%")
print("\nПолученные значения хорошо согласуются с аналитическими расчетами,")
print("что подтверждает адекватность разработанной имитационной модели.")

ЗАДАНИЕ №3.1: Сопоставление с аналитическими результатами

АНАЛИТИЧЕСКИЕ РЕЗУЛЬТАТЫ (на основе теории массового обслуживания):
--------------------------------------------------
Коэффициент загрузки системы (ρ = λ/μ) = 2.0000
Вероятность простоя системы (P0) = 0.2000
Вероятность отказа в обслуживании (Pf) = 0.4000
Q (Относительная пропускная способность) = 0.6000
A (Абсолютная пропускная способность) = 0.1200 [1/сек]
k (Коэффициент загрузки каналов) = 1.2000

СРАВНЕНИЕ РЕЗУЛЬТАТОВ:
Показатель                          Моделирование   Аналитика       Отклонение     
--------------------------------------------------------------------------------
Q (Относительная пропускная способность) 0.5900          0.6000          1.67          %
A (Абсолютная пропускная способность) 0.1180          0.1200          1.67          %
k (Коэффициент загрузки каналов)    1.3014          1.2000          8.45          %
Вероятность отказа (Pf)             0.4100          0.4000          2.50          %
Вероя

### Exercise 3.2

In [27]:
# Cell 6: ЗАДАНИЕ №3.2 - Исследование зависимости от времени моделирования
print("=" * 70)
print("ЗАДАНИЕ №3.2: Исследование зависимости от времени моделирования")
print("=" * 70)

# Different simulation times
T_values = [1000, 2000, 3000, 5000]
results_list = []

# Run simulations for different T values
for T_val in T_values:
    sim_result = simulate_two_channel_system(lambda_rate, mu, T_val)
    
    # Calculate additional metrics for the table
    Tr1_T = sim_result['p_idle1']
    Tr2_T = sim_result['p_idle2']
    load_coefficient = sim_result['k']  # This is already the load coefficient
    
    results_list.append({
        'T': T_val,
        'Pf': sim_result['p_reject'],
        'Q': sim_result['Q'],
        'A': sim_result['A'],
        'k': sim_result['k'],
        'Tr1_T': Tr1_T,
        'Tr2_T': Tr2_T,
        'load_coefficient_alt': 2 - (Tr1_T + Tr2_T),  # Alternative calculation
        'processed': sim_result['processed'],
        'rejected': sim_result['rejected']
    })

# Create DataFrame for better visualization
df_results = pd.DataFrame(results_list)

# Calculate analytical values for comparison
analytical_Pf = analytical['Pf']
analytical_Q = analytical['Q']
analytical_A = analytical['A']
analytical_k = analytical['k']

print("\n" + "=" * 120)
print("ТАБЛИЦА 1: Результаты исследования в зависимости от времени моделирования")
print("=" * 120)
print(f"{'№':<3} {'T [сек]':<10} {'Pf':<12} {'Q':<12} {'A':<12} {'k':<12} {'Tr1/T':<12} {'Tr2/T':<12} {'Обраб.':<8} {'Отказы':<8}")
print("-" * 120)

for i, row in df_results.iterrows():
    print(f"{i+1:<3} {row['T']:<10} {row['Pf']:<12.4f} {row['Q']:<12.4f} {row['A']:<12.4f} {row['k']:<12.4f} {row['Tr1_T']:<12.4f} {row['Tr2_T']:<12.4f} {row['processed']:<8} {row['rejected']:<8}")

print("-" * 120)
print(f"{'Анал.':<3} {'':<10} {analytical_Pf:<12.4f} {analytical_Q:<12.4f} {analytical_A:<12.4f} {analytical_k:<12.4f} {'-':<12} {'-':<12} {'-':<8} {'-':<8}")
print("=" * 120)

print("\n" + "=" * 70)
print("АНАЛИЗ СХОДИМОСТИ:")
print("=" * 70)
print("С увеличением времени моделирования T результаты приближаются к аналитическим значениям:")
print("-" * 70)
print(f"{'T [сек]':<10} {'Q (модель)':<15} {'Q (аналит)':<15} {'Отклонение Q':<15} {'k (модель)':<15} {'Отклонение k':<15}")
print("-" * 70)

for i, row in df_results.iterrows():
    error_Q = abs(row['Q'] - analytical_Q) / analytical_Q * 100
    error_k = abs(row['k'] - analytical_k) / analytical_k * 100
    print(f"{row['T']:<10} {row['Q']:<15.4f} {analytical_Q:<15.4f} {error_Q:<14.2f}% {row['k']:<15.4f} {error_k:<14.2f}%")

ЗАДАНИЕ №3.2: Исследование зависимости от времени моделирования

ТАБЛИЦА 1: Результаты исследования в зависимости от времени моделирования
№   T [сек]    Pf           Q            A            k            Tr1/T        Tr2/T        Обраб.   Отказы  
------------------------------------------------------------------------------------------------------------------------
1   1000.0     0.3679       0.6321       0.1220       1.2127       0.3277       0.4596       122.0    71.0    
2   2000.0     0.3678       0.6322       0.1255       1.2416       0.3123       0.4461       251.0    146.0   
3   3000.0     0.3957       0.6043       0.1207       1.1671       0.3257       0.5072       362.0    237.0   
4   5000.0     0.4037       0.5963       0.1170       1.2246       0.3258       0.4496       585.0    396.0   
------------------------------------------------------------------------------------------------------------------------
Анал.            0.4000       0.6000       0.1200       1.2000  